# Data Preparation

In [64]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from config import (
  UNDERSTANDING_REGENCIES_CSV,
  FEATURE_EVALUATION_JSON,
  FEATURE_SELECTION_JSON,
  PREPARED_REGENCIES_CSV
)

In [65]:
MAX_VIF = 10.0
MIN_CV = 10.0
SKEWNESS_THRESHOLD = 2.0

In [66]:
print(f"Parameter MAX_VIF            : {MAX_VIF}")
print(f"Parameter MIN_CV             : {MIN_CV}%")
print(f"Parameter SKEWNESS_THRESHOLD : {SKEWNESS_THRESHOLD}")

Parameter MAX_VIF            : 10.0
Parameter MIN_CV             : 10.0%
Parameter SKEWNESS_THRESHOLD : 2.0


In [67]:
df_reg = pd.read_csv(UNDERSTANDING_REGENCIES_CSV, dtype={
  'province_id': str,
  'regency_no': str,
  'regency_name': str
})

## Imputasi Nilai Hilang (kNN Imputer)

In [68]:
numeric_cols = df_reg.select_dtypes('number').columns

print("Jumlah Imputasi:")
print(df_reg[numeric_cols].isna().sum())

df_reg[numeric_cols] = KNNImputer(n_neighbors=5).fit_transform(df_reg[numeric_cols])

Jumlah Imputasi:
total_koperasi      0
koperasi_nib        0
koperasi_npwp       0
koperasi_rat        0
simpanan_pokok      0
simpanan_wajib      0
volume_transaksi    0
nilai_transaksi     0
dtype: int64


## Pemilihan Fitur

In [69]:
feature_config = json.load(open(FEATURE_EVALUATION_JSON, 'r', encoding='utf-8'))

vif_dict = feature_config.get('vif', {})
cv_dict = feature_config.get('variability_cv', {})
skew_dict = feature_config.get('skewness', {})

In [70]:
selected_features = []
eliminated_features = []

for feat, vif_val in vif_dict.items():
  cv_val = cv_dict.get(feat, 100.0)
  
  if vif_val <= MAX_VIF and cv_val >= MIN_CV:
    selected_features.append(feat)
  else:
    eliminated_features.append(feat)

log_transform_features = []

for feat in selected_features:
  skew_val = skew_dict.get(feat, 0)
  
  if skew_val >= SKEWNESS_THRESHOLD:
    log_transform_features.append(feat)

print(f"Fitur Terpilih         : {selected_features}")
print(f"Fitur Dieliminasi      : {eliminated_features}")
print(f"Fitur Transformasi Log : {log_transform_features}")

Fitur Terpilih         : ['koperasi_nib', 'koperasi_rat', 'simpanan_pokok', 'simpanan_wajib', 'volume_transaksi', 'nilai_transaksi']
Fitur Dieliminasi      : ['total_koperasi', 'koperasi_npwp']
Fitur Transformasi Log : ['simpanan_pokok', 'simpanan_wajib', 'volume_transaksi', 'nilai_transaksi']


## Transformasi & Standardisasi

### Transformasi Logaritmik (Log1p)

In [71]:
features_present = []

for col in selected_features:
  if col in df_reg.columns:
    features_present.append(col)

X_df = df_reg[features_present].copy()

for col in log_transform_features:
  if col in X_df.columns:
    X_df[col] = np.log1p(X_df[col].clip(lower=0))

print(X_df.describe().T.to_markdown())

|                  |   count |      mean |       std |   min |     25% |      50% |      75% |      max |
|:-----------------|--------:|----------:|----------:|------:|--------:|---------:|---------:|---------:|
| koperasi_nib     |     514 | 118.325   | 100.51    |     0 | 46      | 86       | 160      | 570      |
| koperasi_rat     |     514 |  97.6206  |  89.5316  |     0 | 34.25   | 71       | 138      | 596      |
| simpanan_pokok   |     514 |  15.2569  |   5.1418  |     0 | 15.4073 | 16.8543  |  17.9287 |  21.5539 |
| simpanan_wajib   |     514 |  13.8853  |   5.37185 |     0 | 13.9255 | 15.6584  |  16.9431 |  20.1092 |
| volume_transaksi |     514 |   5.73138 |   5.14031 |     0 |  0      |  6.96484 |  10.5982 |  14.2969 |
| nilai_transaksi  |     514 |  11.2551  |   9.33017 |     0 |  0      | 16.7176  |  19.552  |  23.4442 |


### Standardisasi Fitur (StandardScaler)

In [72]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values)

scaled_cols = []

for c in features_present:
  scaled_cols.append(f"scaled_{c}")

df_reg[scaled_cols] = X_scaled

print(f"Total Kolom Fitur Terstandarisasi Ditambahkan: {len(scaled_cols)}")

Total Kolom Fitur Terstandarisasi Ditambahkan: 6


## Penyimpanan Hasil

In [73]:
df_reg.to_csv(PREPARED_REGENCIES_CSV, index=False)
print(f"File Prepared Kab/Kota disimpan di       : {PREPARED_REGENCIES_CSV}")

File Prepared Kab/Kota disimpan di       : d:\kopdes\artifact\2_preparation\prepared_regencies.csv


In [74]:
feature_selection = {
  'selected_features': selected_features,
  'eliminated_features': eliminated_features,
  'log_transform_features': log_transform_features,
  'scaled_feature_columns': scaled_cols
}

json.dump(feature_selection, open(FEATURE_SELECTION_JSON, 'w', encoding='utf-8'), indent=2)
print(f'Hasil Feature Selection disimpan di      : {FEATURE_SELECTION_JSON}')


Hasil Feature Selection disimpan di      : d:\kopdes\artifact\2_preparation\feature_selection.json
